# 第 10 章 · 指令微调 SFT

> 📄 配套动画网页：`../10_sft.html`　（先用浏览器打开，把原理动画看一遍再来写代码）

**所属部分**：第三部分 · 训练

预训练后的模型会「接话」但不会「听话」。SFT 教它遵守对话格式，关键技巧是——损失只算回答部分。

---


## 🎯 本章目标

- 理解 chat template 的作用
- 掌握 `apply_chat_template` 的用法
- 理解 loss mask 与 `ignore_index=-100`
- 能实现 `generate_labels`（只对 assistant 回答计损失）

---


## 📖 背景：为什么要有 loss mask

如果把用户的问题也拿来算损失，模型会去「学习生成用户的问题」——这是浪费甚至有害的。

正确做法：把问题部分的标签设成 `-100`，交叉熵会自动忽略。

```python
labels = [-100, -100, ..., 回答的 token id, ...]
loss = F.cross_entropy(logits, labels, ignore_index=-100)
```

MiniMind 的实现思路：先分词出 `bos_id = tokenizer("<|im_start|>assistant\n")`，
然后扫描序列，找到每个 `bos_id` 的位置，从它之后开始把标签设为原文 id，直到遇到 `eos_id`。

---



## 🖼 官方配图

![MiniMind SFT 阶段损失曲线](../images/sft_loss.jpg)

*MiniMind SFT 阶段损失曲线*

---


# 第一部分 · 基础函数演示

> 这一部分**可以直接运行**。先把本章要用到的 Python / PyTorch 函数和方法摸熟，后面自己写的时候才不会卡在 API 上。


### 10.1 对话数据长什么样

In [ ]:
import json

conversations = [
    {"role": "system", "content": "你是一个知识丰富的AI，尽力为用户提供准确的信息。"},
    {"role": "user", "content": "1+1等于几？"},
    {"role": "assistant", "content": "1+1=2。"},
]
print(json.dumps(conversations, ensure_ascii=False, indent=2))

### 10.2 `apply_chat_template`

分词器自带模板，会把对话列表渲染成带特殊 token 的字符串。

In [ ]:
from pathlib import Path

tok_path = Path("../data")
has_tok = (tok_path / "tokenizer_config.json").exists()

if has_tok:
    try:
        from transformers import AutoTokenizer
        tok = AutoTokenizer.from_pretrained(str(tok_path))
        prompt = tok.apply_chat_template(conversations, tokenize=False, add_generation_prompt=False)
        print(prompt)
        print("--- 编码后 ---")
        print(tok(prompt).input_ids[:30])
    except ImportError:
        has_tok = False
        print("未安装 transformers —— 请先执行 `uv sync`。")

if not has_tok:
    # 没有官方分词器时，用一段手工模板演示同样的结构
    prompt = ("<|im_start|>system\n你是一个AI助手<|im_end|>\n"
              "<|im_start|>user\n1+1等于几？<|im_end|>\n"
              "<|im_start|>assistant\n1+1=2。<|im_end|>\n")
    print("（未找到官方分词器，使用手工模板演示）\n")
    print(prompt)

### 10.3 构造 loss mask 的基本工具

In [ ]:
n = 10
labels = [-100] * n
print("初始全掩码:", labels)

# 假设从第 6 个位置开始是回答
for j in range(6, 10):
    labels[j] = j + 100        # 用假 id 代替
print("只标记回答部分:", labels)

### 10.4 子序列查找：找到 `bos_id` 出现的位置

这是 `generate_labels` 的核心——在 token 序列里找「assistant 开始」标记。

In [ ]:
ids = [1, 5, 6, 7, 2, 1, 8, 9, 2, 3]
bos_id = [1]
eos_id = [2]

i, spans = 0, []
while i < len(ids):
    if ids[i:i + len(bos_id)] == bos_id:
        start = i + len(bos_id)
        end = start
        while end < len(ids) and ids[end:end + len(eos_id)] != eos_id:
            end += 1
        spans.append((start, end + len(eos_id)))
        i = end + len(eos_id)
    else:
        i += 1
print("回答区间:", spans, " <- 每个区间内的 token 才计损失")

# 第二部分 · 动手实现（只给提示，留白给你）

> 下面每一节都**没有答案**。请根据文字提示 + 第一部分演示过的函数，自己从零写出来。
> 写不出来时，先回去看第一部分的演示，再对照网页里的动画想清楚数据形状。

SFT 的核心就是 `generate_labels`。请自己写出来。

> 💡 每个留白代码块后面都跟了一个**测试块**。先写代码，再运行测试块。


In [ ]:
# ════════════════════ 自测工具 ════════════════════
# 每个留白代码块后面都跟一个测试块。先在上面写代码，再运行测试块。
# 没写之前运行也不会报错，只会提示「未实现」。
import torch
import torch.nn.functional as F

_PASS, _FAIL, _SKIP = [], [], []

def check(name, fn):
    """运行一个测试。捕获异常并打印结果，不会中断后面的测试。"""
    try:
        fn()
        print(f"✅ 通过   {name}")
        _PASS.append(name)
    except NameError as e:
        print(f"⏸  未实现 {name} —— {e}")
        _SKIP.append(name)
    except AssertionError as e:
        print(f"❌ 失败   {name} —— {e}")
        _FAIL.append(name)
    except Exception as e:
        print(f"❌ 出错   {name} —— {type(e).__name__}: {e}")
        _FAIL.append(name)

def close(a, b, tol=1e-4, msg=""):
    """断言两个张量 / 数字足够接近"""
    a, b = torch.as_tensor(a).float().reshape(-1), torch.as_tensor(b).float().reshape(-1)
    assert a.numel() == b.numel(), f"元素个数不一致 {a.numel()} vs {b.numel()} {msg}"
    assert torch.allclose(a, b, atol=tol), f"数值不接近（容差 {tol}）{msg}"

def same(a, b, msg=""):
    """断言两个张量完全相同"""
    a, b = torch.as_tensor(a), torch.as_tensor(b)
    assert a.shape == b.shape, f"形状不一致 {tuple(a.shape)} vs {tuple(b.shape)} {msg}"
    assert torch.equal(a, b), f"内容不相同 {msg}"

def summary():
    print()
    print(f"通过 {len(_PASS)} · 失败 {len(_FAIL)} · 未实现 {len(_SKIP)}")
    if _FAIL: print("需要修的：", "、".join(_FAIL))
    if _SKIP: print("还没写的：", "、".join(_SKIP))
    if _PASS and not _FAIL and not _SKIP:
        print("本章全部通过 🎉")

print("自测工具已就绪 —— 每写完一个留白，就运行它下面的测试块。")

### 10.5 实现 `generate_labels`

**要求**：写一个**独立函数**（方便单独测试），把一条已经拼好模板并分词的序列变成 loss mask 标签。

```python
def generate_labels(input_ids, bos_id, eos_id, max_length):
    # input_ids 是 Python list；返回同长度的 labels，默认全 -100
```

规则：
- 每当匹配到子序列 `bos_id`（即 `<|im_start|>assistant\n`），就从它**之后**开始，
  把标签设为对应的 `input_ids`，直到遇到 `eos_id`（含 eos）
- 边界不要越界；超过 `max_length` 的部分截断
- 循环要能处理**多轮对话**（跳过一段后继续往后扫描）

**验证**：打印出哪些位置计入了损失，确认只有 assistant 的回答部分。

In [ ]:
# ======== 你的实现：generate_labels ========
#
# 【要求】
#   - 签名：generate_labels(input_ids, bos_id, eos_id, max_length)
#   - labels = [-100] * len(input_ids)
#   - 用 while 扫描，用切片比较匹配 bos_id 子序列
#   - 匹配到后，从 start 开始把 input_ids 复制进 labels，直到遇到 eos_id
#   - 跳过整个区间后，i 前进到 eos 之后继续扫描下一轮
#
# 【可用到的函数 / 方法】
#   - 列表切片比较 `input_ids[i:i+len(bos_id)] == bos_id`
#   - `min(end + len(eos_id), max_length)`
#
# 提示：先想清楚输入输出的形状，再动手。
#
# ↓↓↓ 在这里写你的代码 ↓↓↓




In [ ]:
# ════════════ 测试：generate_labels ════════════
# 先在上面把代码写出来，再运行这一格。
def _t():
    ids = [1, 5, 6, 7, 2, 1, 8, 9, 2, 3]
    labels = generate_labels(ids, bos_id=[1], eos_id=[2], max_length=len(ids))
    assert len(labels) == len(ids), f"labels 长度应与 input_ids 一致，得到 {len(labels)}"
    assert labels[0] == -100, "标记之外的 token 应为 -100"
    assert labels[1:5] == [5, 6, 7, 2], f"第一个回答区间应为 [5, 6, 7, 2]，得到 {labels[1:5]}"
    assert labels[5] == -100, "bos_id 自身不应计入损失"
    assert labels[6:9] == [8, 9, 2], f"第二个回答区间应为 [8, 9, 2]，得到 {labels[6:9]}"
    assert labels[9] == -100, "尾部未匹配的 token 应为 -100"

check("generate_labels 只标记 assistant 回答", _t)

### 10.6 实现 `SFTDataset`

**要求**：写出这个 Dataset 类。

```python
class SFTDataset(Dataset):
    def __init__(self, jsonl_path, tokenizer, max_length=1024):
        # 用 json 逐行读 jsonl，每行形如 {"conversations": [...]}
        # 预先算好 self.bos_id = tokenizer(f"{tokenizer.bos_token}assistant\n",
        #                                   add_special_tokens=False).input_ids
        #        self.eos_id = tokenizer(f"{tokenizer.eos_token}\n",
        #                                add_special_tokens=False).input_ids

    def __getitem__(self, index):
        # 1. 取出 conversations，套模板得到 prompt
        # 2. input_ids = tokenizer(prompt).input_ids[:max_length]
        # 3. 补齐 pad 到 max_length
        # 4. labels = generate_labels(input_ids, self.bos_id, self.eos_id, max_length)
        # 5. 返回两个 torch.long 张量
```

> `generate_labels` 是上面那节写的**模块级函数**，直接调用即可。

In [ ]:
# ======== 你的实现：SFTDataset ========
#
# 【要求】
#   - 继承 torch.utils.data.Dataset
#   - __init__ 里用 json 逐行读 jsonl，并预先算好 self.bos_id / self.eos_id
#   - __getitem__ 里套模板（可自己拼一个简单模板，或用 apply_chat_template）
#   - 分词、截断到 max_length、补齐 pad
#   - 调用上面写好的 generate_labels 得到 labels
#   - 返回 (input_ids, labels)，都是 torch.long 张量
#
# 【可用到的函数 / 方法】
#   - `tokenizer.apply_chat_template(convs, tokenize=False)`
#   - `tokenizer(prompt).input_ids`
#   - 上面写好的 `generate_labels`
#
# 提示：先想清楚输入输出的形状，再动手。
#
# ↓↓↓ 在这里写你的代码 ↓↓↓




In [ ]:
# ════════════ 测试：SFTDataset ════════════
# 先在上面把代码写出来，再运行这一格。
def _t():
    import json, tempfile, os

    class FakeTok:
        pad_token_id = 0
        bos_token, eos_token = "<A>", "<E>"
        def apply_chat_template(self, convs, tokenize=False, add_generation_prompt=False):
            out = ""
            for m in convs:
                if m["role"] == "assistant":
                    out += "<A>assistant\n" + m["content"] + "<E>\n"
                else:
                    out += m["content"]
            return out
        def __call__(self, text, add_special_tokens=True, max_length=None, truncation=False):
            ids, i = [], 0
            while i < len(text):
                if text.startswith("<A>", i):
                    ids.append(1); i += 3
                elif text.startswith("<E>", i):
                    ids.append(2); i += 3
                else:
                    ids.append(3 + (ord(text[i]) % 20)); i += 1
            return type("Enc", (), {"input_ids": ids[:max_length] if max_length else ids})()

    path = os.path.join(tempfile.gettempdir(), "mm_sft_ds_test.jsonl")
    with open(path, "w", encoding="utf-8") as f:
        f.write(json.dumps({"conversations": [
            {"role": "user", "content": "hi"},
            {"role": "assistant", "content": "ok"},
        ]}, ensure_ascii=False) + "\n")

    ds = SFTDataset(path, FakeTok(), max_length=32)
    x, y = ds[0]
    assert tuple(x.shape) == (32,), f"input_ids 形状应为 (32,)，得到 {tuple(x.shape)}"
    assert tuple(y.shape) == (32,), f"labels 形状应为 (32,)，得到 {tuple(y.shape)}"
    valid = y != -100
    assert (~valid).any(), "问题部分应该被掩码成 -100"
    assert valid.any(), "assistant 的回答部分应该计入损失"
    same(y[valid], x[valid], msg="计入损失的标签应等于 input_ids 本身")
    assert valid.sum().item() >= 2, f"计入损失的 token 太少（{valid.sum().item()} 个），掩码可能不对"

check("SFTDataset 只对 assistant 回答计损失", _t)

In [ ]:
summary()

# 第三部分 · 自测与思考

- [ ] 能解释为什么问题部分要设成 -100
- [ ] `bos_id` 为什么包含 `assistant\n` 而不只是 `<|im_start|>`？
- [ ] 你的 `generate_labels` 在 eos 处正确闭合了吗？
- [ ] 一条样本里大约有多少比例的 token 参与损失？

**思考题**：如果多轮对话里每一轮 assistant 都要计损失，你的实现能处理吗？（提示：`while` 循环会继续往后扫描）
